# Collaborating Agents in Practice

A **multi-agent system** is not "more LLM calls" — it is **multiple roles** with explicit **handoffs**, **shared context**, and **termination rules**. In production, collaboration fails when agents talk past each other: duplicate work, contradictory answers, or hidden state no one can audit.

```
  User ticket / release task
         │
         ▼
  ┌──────────────┐   handoff envelope   ┌──────────────┐
  │   Triage     │ ───────────────────► │   Policy     │
  │   Agent      │                      │   Agent      │
  └──────┬───────┘                      └──────┬───────┘
         │ shared board                       │
         ▼                                      ▼
  ┌──────────────┐   critique / approve  ┌──────────────┐
  │   Order      │ ◄─────────────────── │   QA Agent   │
  │   Agent      │                      │              │
  └──────────────┘                      └──────────────┘
```

This notebook implements **collaboration mechanics** in plain Python for two scenarios:

- **ShopCo Support Hub** — triage → specialists → QA on a shared case board
- **ReleaseFlow** — researcher → writer → gatekeeper on a release draft

You will see **handoff envelopes**, **supervisor routing**, **parallel fan-out**, and **collaboration traces** — the plumbing that makes multi-agent demos work in real systems.

In [ ]:
"""
Collaborating agents — ShopCo + ReleaseFlow lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
MAX_COLLAB_ROUNDS = 6


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days with receipt. [pol-returns-01]",
    "shipping": "Free shipping on orders over $50. [pol-shipping-02]",
    "billing": "Charges appear as SHOPCO* on statements. [pol-billing-03]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "customer": "Alice"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "customer": "Bob"},
}

CHANGELOG = [
    {"version": "2.4.0", "item": "Added bulk export endpoint"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

STYLE_GUIDE = {
    "must_include": ["version number", "breaking changes", "support channel"],
    "support_channel": "#releases",
}

print("Collaboration lab ready.")

---

## 1. Collaboration Styles in Production

| Style | How agents coordinate | Example |
|-------|----------------------|---------|
| **Supervisor routing** | Lead picks next worker | Triage → policy or order |
| **Sequential handoff** | A finishes, passes envelope to B | Researcher → writer |
| **Shared blackboard** | All read/write structured board | Case facts on `CaseBoard` |
| **Parallel fan-out** | Independent workers, merge results | CI checks in parallel |
| **Critique loop** | Producer + reviewer until approve | QA agent gates reply |

Most real systems **combine** styles — supervisor + board + critique is common for support hubs.

---

## 2. Agent Roles and Contracts

Each agent declares:

1. **Role id** — stable name for traces
2. **Inputs** — what it reads from shared state
3. **Outputs** — what it writes back
4. **Handoff target** — who may run next (optional)

Contracts prevent agents from silently overwriting each other's fields.

In [ ]:
@dataclass
class AgentContract:
    agent_id: str
    reads: list[str]
    writes: list[str]
    description: str


SHOPCO_CONTRACTS = [
    AgentContract("triage", ["customer_message"], ["intent", "order_id", "customer_name"], "Classify ticket"),
    AgentContract("policy", ["intent", "customer_message"], ["policy_citations", "draft_answer"], "Policy lookup"),
    AgentContract("order", ["order_id"], ["order_status", "draft_answer"], "Order lookup"),
    AgentContract("qa", ["draft_answer", "policy_citations"], ["final_answer", "approved"], "Quality gate"),
]

RELEASE_CONTRACTS = [
    AgentContract("researcher", ["version"], ["changelog_bullets"], "Gather facts"),
    AgentContract("writer", ["changelog_bullets"], ["draft_announcement"], "Draft comms"),
    AgentContract("gatekeeper", ["draft_announcement"], ["final_announcement", "approved"], "Style compliance"),
]

print("ShopCo agents:", [c.agent_id for c in SHOPCO_CONTRACTS])

---

## 3. Handoff Envelope — The Collaboration Wire Format

Agents pass **envelopes**, not raw strings. An envelope carries intent, payload, and provenance.

In [ ]:
@dataclass
class HandoffEnvelope:
    envelope_id: str
    case_id: str
    from_agent: str
    to_agent: str
    intent: str
    payload: dict[str, Any]
    ts: str = field(default_factory=utc_now)


@dataclass
class CollaborationTrace:
    case_id: str
    events: list[dict[str, Any]] = field(default_factory=list)

    def log(self, event_type: str, agent: str, detail: dict[str, Any]) -> None:
        self.events.append({
            "ts": utc_now(),
            "type": event_type,
            "agent": agent,
            **detail,
        })


env = HandoffEnvelope(
    envelope_id="env-001", case_id="CASE-1",
    from_agent="triage", to_agent="policy",
    intent="needs_return_policy", payload={"topic": "returns"},
)
print(pretty(env))

---

## 4. Shared Case Board — ShopCo Blackboard

In [ ]:
@dataclass
class CaseBoard:
    case_id: str
    customer_message: str = ""
    customer_name: str | None = None
    order_id: str | None = None
    intent: str | None = None
    policy_citations: list[str] = field(default_factory=list)
    order_status: str | None = None
    draft_answer: str | None = None
    final_answer: str | None = None
    approved: bool = False

    def snapshot(self) -> dict[str, Any]:
        return {k: getattr(self, k) for k in self.__dataclass_fields__}


board = CaseBoard(case_id="CASE-DEMO", customer_message="Can Alice return ORD-1001?")
print(board.snapshot())

---

## 5. ShopCo Specialist Agents

In [ ]:
def triage_agent(board: CaseBoard) -> HandoffEnvelope | None:
    msg = board.customer_message
    name_m = re.search(r"([A-Za-z]+)", msg)
    if "alice" in msg.lower():
        board.customer_name = "Alice"
    elif name_m and "return" not in msg.lower():
        board.customer_name = name_m.group(1)
    oid = re.search(r"ORD-[0-9]{4}", msg.upper())
    if oid:
        board.order_id = oid.group(0)
    low = msg.lower()
    if "return" in low:
        board.intent = "return_question"
        return HandoffEnvelope(
            f"env-{uuid.uuid4().hex[:6]}", board.case_id,
            "triage", "policy", "needs_return_policy", {"topic": "returns"},
        )
    if "order" in low or "ship" in low or board.order_id:
        board.intent = "order_status"
        return HandoffEnvelope(
            f"env-{uuid.uuid4().hex[:6]}", board.case_id,
            "triage", "order", "needs_order_lookup", {"order_id": board.order_id},
        )
    board.intent = "general"
    board.draft_answer = "Thanks for contacting ShopCo. How can we help?"
    return HandoffEnvelope(
        f"env-{uuid.uuid4().hex[:6]}", board.case_id,
        "triage", "qa", "needs_qa", {},
    )


def policy_agent(board: CaseBoard) -> HandoffEnvelope:
    topic = "returns" if board.intent == "return_question" else "shipping"
    cite = POLICY_SNIPPETS[topic]
    board.policy_citations.append(cite)
    board.draft_answer = f"Per policy: {cite}"
    if board.order_id:
        board.draft_answer += f" (order {board.order_id} on file)"
    return HandoffEnvelope(
        f"env-{uuid.uuid4().hex[:6]}", board.case_id,
        "policy", "qa", "needs_qa", {},
    )


def order_agent(board: CaseBoard) -> HandoffEnvelope:
    oid = board.order_id or ""
    rec = ORDERS_DB.get(oid, {"status": "unknown"})
    board.order_status = rec["status"]
    board.draft_answer = f"Order {oid or 'N/A'} status: {board.order_status}."
    return HandoffEnvelope(
        f"env-{uuid.uuid4().hex[:6]}", board.case_id,
        "order", "qa", "needs_qa", {},
    )


def qa_agent(board: CaseBoard) -> HandoffEnvelope | None:
    draft = board.draft_answer or ""
    issues: list[str] = []
    if board.intent == "return_question" and not board.policy_citations:
        issues.append("missing policy citation")
    if "deal with it" in draft.lower():
        issues.append("harsh tone")
    if len(draft) < 10:
        issues.append("too short")
    if issues:
        board.draft_answer = draft + f" {POLICY_SNIPPETS['returns']}"
        board.approved = "missing policy citation" not in issues
    else:
        board.approved = True
    board.final_answer = board.draft_answer
    return None  # terminal


SHOPCO_AGENTS: dict[str, Callable[[CaseBoard], HandoffEnvelope | None]] = {
    "triage": triage_agent,
    "policy": policy_agent,
    "order": order_agent,
    "qa": qa_agent,
}

---

## 6. Supervisor — Routes Handoffs Until Done

In [ ]:
@dataclass
class CollaborationResult:
    case_id: str
    board: CaseBoard
    agents_run: list[str]
    handoffs: list[HandoffEnvelope]
    trace: CollaborationTrace


class Supervisor:
    def __init__(self, agents: dict[str, Callable[[CaseBoard], HandoffEnvelope | None]]) -> None:
        self.agents = agents

    def run(self, board: CaseBoard, start_agent: str = "triage") -> CollaborationResult:
        trace = CollaborationTrace(case_id=board.case_id)
        agents_run: list[str] = []
        handoffs: list[HandoffEnvelope] = []
        current = start_agent
        rounds = 0

        while current and rounds < MAX_COLLAB_ROUNDS:
            rounds += 1
            agent_fn = self.agents[current]
            trace.log("agent_start", current, {"round": rounds})
            envelope = agent_fn(board)
            agents_run.append(current)
            trace.log("agent_done", current, {"board": board.snapshot()})

            if envelope is None:
                break
            handoffs.append(envelope)
            trace.log("handoff", current, {"to": envelope.to_agent, "intent": envelope.intent})
            current = envelope.to_agent

        return CollaborationResult(board.case_id, board, agents_run, handoffs, trace)


supervisor = Supervisor(SHOPCO_AGENTS)
shopco_result = supervisor.run(CaseBoard(
    case_id="CASE-1001",
    customer_message="Can Alice return ORD-1001?",
))
print("Agents:", shopco_result.agents_run)
print("Final:", shopco_result.board.final_answer)
print("Approved:", shopco_result.board.approved)

---

## 7. Demo — Order Status Collaboration Path

In [ ]:
order_result = supervisor.run(CaseBoard(
    case_id="CASE-1002",
    customer_message="Where is order ORD-1002?",
))
print("Path:", " → ".join(order_result.agents_run))
print("Answer:", order_result.board.final_answer)
print("\nHandoffs:")
for h in order_result.handoffs:
    print(f"  {h.from_agent} → {h.to_agent} ({h.intent})")

---

## 8. ReleaseFlow — Sequential Handoff Chain

Researcher gathers facts → writer drafts → gatekeeper approves. Each agent only sees fields in its **contract**.

In [ ]:
@dataclass
class ReleaseBoard:
    release_id: str
    version: str = "2.4.0"
    changelog_bullets: list[str] = field(default_factory=list)
    draft_announcement: str | None = None
    final_announcement: str | None = None
    approved: bool = False


def researcher_agent(rb: ReleaseBoard) -> str:
    rb.changelog_bullets = [
        f"- {c['item']}" for c in CHANGELOG if c["version"] == rb.version
    ]
    return "writer"


def writer_agent(rb: ReleaseBoard) -> str:
    bullets = "\n".join(rb.changelog_bullets)
    rb.draft_announcement = (
        f"Release {rb.version} is now available.\n"
        f"Highlights:\n{bullets}\n"
        f"Breaking changes: see deprecated webhook note above."
    )
    return "gatekeeper"


def gatekeeper_agent(rb: ReleaseBoard) -> str | None:
    draft = rb.draft_announcement or ""
    missing = [k for k in STYLE_GUIDE["must_include"] if k.replace(" ", "") not in draft.lower().replace(" ", "")]
    if "support" not in draft.lower():
        draft += f"\nQuestions? Reach us on {STYLE_GUIDE['support_channel']}."
        rb.draft_announcement = draft
        missing = [m for m in missing if m != "support channel"]
    rb.approved = len(missing) == 0
    rb.final_announcement = draft if rb.approved else draft + "\n[NEEDS REVISION]"
    return None


RELEASE_CHAIN: list[tuple[str, Callable[[ReleaseBoard], str | None]]] = [
    ("researcher", researcher_agent),
    ("writer", writer_agent),
    ("gatekeeper", gatekeeper_agent),
]


def run_sequential_chain(board: ReleaseBoard) -> dict[str, Any]:
    trace = CollaborationTrace(case_id=board.release_id)
    agents_run: list[str] = []
    for agent_id, fn in RELEASE_CHAIN:
        trace.log("agent_start", agent_id, {})
        next_agent = fn(board)
        agents_run.append(agent_id)
        trace.log("agent_done", agent_id, {"approved": board.approved})
        if next_agent is None:
            break
    return {"agents_run": agents_run, "board": board, "trace_events": len(trace.events)}


release_result = run_sequential_chain(ReleaseBoard(release_id="REL-240"))
print("Agents:", release_result["agents_run"])
print("\nAnnouncement:\n", release_result["board"].final_announcement)

---

## 9. Parallel Collaboration — Fan-Out and Merge

When subtasks are **independent**, run workers in parallel (simulated here) and merge.

In [ ]:
def security_worker(_: str) -> dict[str, str]:
    return {"domain": "security", "finding": "1 medium CVE in dev dependency", "status": "warn"}


def tests_worker(_: str) -> dict[str, str]:
    return {"domain": "tests", "finding": "412 tests passed", "status": "pass"}


def docs_worker(version: str) -> dict[str, str]:
    return {"domain": "docs", "finding": f"changelog has {len(CHANGELOG)} items for {version}", "status": "pass"}


def parallel_collaboration(version: str) -> dict[str, Any]:
    workers = [security_worker, tests_worker, docs_worker]
    results = [w(version) for w in workers]
    blocking = [r for r in results if r["status"] == "fail"]
    gate = "block" if blocking else "proceed"
    return {"pattern": "parallel_fan_out", "gate": gate, "worker_results": results}


print(pretty(parallel_collaboration("2.4.0")))

---

## 10. Critique Loop — Revise Until Approved

Writer and gatekeeper may loop when the first draft fails style checks.

In [ ]:
def score_announcement(text: str) -> tuple[int, list[str]]:
    issues = []
    score = 0
    if re.search(r"2\.4\.0|v?\d+\.\d+\.\d+", text):
        score += 30
    else:
        issues.append("missing version")
    if "breaking" in text.lower():
        score += 25
    else:
        issues.append("missing breaking changes")
    if "#releases" in text or "support" in text.lower():
        score += 25
    else:
        issues.append("missing support channel")
    if len(text.split()) >= 20:
        score += 20
    else:
        issues.append("too brief")
    return score, issues


def critique_loop(version: str, max_rounds: int = 3) -> dict[str, Any]:
    rb = ReleaseBoard(release_id="REL-CRITIQUE", version=version)
    researcher_agent(rb)
    rounds: list[dict[str, Any]] = []
    for i in range(max_rounds):
        writer_agent(rb)
        score, issues = score_announcement(rb.draft_announcement or "")
        rounds.append({"round": i + 1, "score": score, "issues": issues})
        if score >= 80:
            gatekeeper_agent(rb)
            break
        rb.draft_announcement = (rb.draft_announcement or "") + f"\nSupport: {STYLE_GUIDE['support_channel']}"
    return {"rounds": rounds, "approved": rb.approved, "final": rb.final_announcement}


print(pretty(critique_loop("2.4.0")))

---

## 11. Failure Handling — Agent Errors and Fallbacks

When a specialist fails, the supervisor should **not** crash the case — route to fallback or human.

In [ ]:
def flaky_order_agent(board: CaseBoard) -> HandoffEnvelope:
    if board.order_id == "ORD-9999":
        raise RuntimeError("Order service timeout")
    return order_agent(board)


def supervisor_with_fallback(board: CaseBoard) -> CollaborationResult:
    trace = CollaborationTrace(case_id=board.case_id)
    agents_run = ["triage"]
    handoffs: list[HandoffEnvelope] = []

    env = triage_agent(board)
    if env and env.to_agent == "order":
        try:
            env2 = flaky_order_agent(board)
            agents_run.append("order")
            handoffs.append(env2)
            qa_agent(board)
            agents_run.append("qa")
        except RuntimeError as exc:
            trace.log("agent_error", "order", {"error": str(exc)})
            board.draft_answer = "Order lookup temporarily unavailable. A human agent will follow up."
            board.final_answer = board.draft_answer
            board.approved = False
            agents_run.append("fallback")
    return CollaborationResult(board.case_id, board, agents_run, handoffs, trace)


fail_result = supervisor_with_fallback(CaseBoard(
    case_id="CASE-FAIL",
    customer_message="Status of ORD-9999?",
))
print("Agents:", fail_result.agents_run)
print("Answer:", fail_result.board.final_answer)

---

## 12. Collaboration Trace — Who Did What

Production systems log every agent turn for debugging and compliance.

In [ ]:
def print_trace(result: CollaborationResult) -> None:
    print(f"Case {result.case_id}")
    for e in result.trace.events:
        print(f"  [{e['type']}] {e['agent']}: { {k: v for k, v in e.items() if k not in ('ts', 'type', 'agent')} }")


print_trace(shopco_result)

---

## 13. Sync vs Async Collaboration

| Mode | When | Trade-off |
|------|------|-----------|
| **Synchronous** | User waiting in chat | Simple traces, higher latency |
| **Async queue** | Long ReleaseFlow jobs | Better throughput, harder UX |
| **Human-in-loop** | QA rejects draft | Pauses until approval |

ShopCo support is usually **sync** with a supervisor. ReleaseFlow often mixes **async parallel checks** with **sync critique loops**.

---

## 14. Choosing a Collaboration Shape

| Problem | Collaboration shape |
|---------|---------------------|
| Disjoint intents | Supervisor routing |
| Fixed pipeline | Sequential handoff |
| Independent checks | Parallel fan-out |
| Quality gate | Critique loop |
| Ambiguous multi-domain ticket | Board + multiple specialists |

Start with **one supervisor and two specialists** before adding more agents.

---

## 15. Optional Live LLM Writer

Set `USE_LIVE_LLM = True` to draft release notes with `gpt-4o-mini`.

In [ ]:
def writer_agent_live(rb: ReleaseBoard) -> str:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0.3)
    resp = llm.invoke([
        SystemMessage(content="Write a concise release announcement. Include version, breaking changes, and #releases support channel."),
        HumanMessage(content=f"Version {rb.version}\nChanges:\n" + "\n".join(rb.changelog_bullets)),
    ])
    rb.draft_announcement = str(resp.content)
    return "gatekeeper"


rb_live = ReleaseBoard(release_id="REL-LIVE", version="2.4.0")
researcher_agent(rb_live)
if USE_LIVE_LLM:
    writer_agent_live(rb_live)
    gatekeeper_agent(rb_live)
    print(rb_live.final_announcement)
else:
    writer_agent(rb_live)
    print("Offline draft:\n", rb_live.draft_announcement)

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Chat-only handoffs | Lost structure | Use `HandoffEnvelope` + board |
| No contracts | Agents overwrite fields | Declare read/write per role |
| Missing trace | Can't debug misroutes | `CollaborationTrace` per case |
| Too many agents | Latency + confusion | Supervisor + 2 specialists first |
| No failure path | User sees 500 | Fallback agent or human queue |
| Unbounded critique loop | Cost spike | `max_rounds` cap |

---

## 17. Quiz

1. What three fields belong in a handoff envelope?
2. When is parallel fan-out preferable to sequential handoff?
3. What does the QA agent add to ShopCo collaboration?
4. How does `AgentContract` reduce conflicts?
5. Why log `CollaborationTrace` events in production?

---

## 18. Summary

**Collaborating agents in practice** require explicit **roles**, **handoff envelopes**, a **shared board**, and a **supervisor** (or chain) that routes work until termination. ShopCo demonstrated triage → specialist → QA; ReleaseFlow showed researcher → writer → gatekeeper with optional critique rounds.

Parallel workers handle independent checks; traces make multi-agent systems **debuggable**. Add agents only when a new role has a clear contract — not because multi-agent sounds impressive.